# Learned noise-aware attention — screening on Colab

Short screening runs for the second contribution. A small CNN estimates a per-pixel noise map from
the low-light input and a learned 1x1 conv fuses it into IG-MSA value features
(`scripts/learned_noise.py`). **Selection uses validation PSNR only** — the held-out test set is never
touched here. Runtime: ~5 h on one T4 (3 runs x 20k iterations).

**Settings:** Colab → Runtime → Change runtime type → GPU (T4). Internet on (clone + gdown).


In [ ]:
import os, sys, subprocess, glob, json, zipfile
from pathlib import Path

import os
!git clone -q https://github.com/Crazycraftie/low-light-enhancement-btp.git /content/btp
os.chdir('/content/btp')
print('HEAD:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())
subprocess.run([sys.executable,'-m','pip','install','-q','einops','gdown','lmdb','yacs','addict','future','natsort','yapf','thop','timm','lpips','scikit-image'], check=True)
!git clone -q --depth 1 https://github.com/caiyuanhao1998/Retinexformer.git baselines/Retinexformer

import torch
assert torch.cuda.is_available(), 'Enable a GPU in Colab: Runtime -> Change runtime type -> T4'
print('GPU:', torch.cuda.get_device_name(0))
OPT = 'baselines/Retinexformer/Options/RetinexFormer_LOL_v1.yml'


## Data + pretrained weights

Use the official LOL-v2-syn weights. Starting from them (rather than from scratch) is what makes a
20k-iteration screen meaningful: the control starts at ~25.7 dB, so any change is attributable to the gate.


In [ ]:
def gdl(fid, out, tries=4):
    out=Path(out); out.parent.mkdir(parents=True,exist_ok=True)
    for _ in range(tries):
        if subprocess.run(['gdown','-q',fid,'-O',str(out)]).returncode==0 and out.exists(): return
    raise RuntimeError('download failed '+str(out))

DATA = Path('data/LOLv2/Synthetic')
if not (DATA/'Train/Low').is_dir():
    gdl('1Ou9EljYZW8o5dbDCf9R34FS8Pd8kEp2U','/content/lolv2.zip')
    subprocess.run(['unzip','-q','-o','/content/lolv2.zip','-d','data'],check=True)
assert len(list((DATA/'Train/Low').glob('*')))==900
assert len(list((DATA/'Test/Low').glob('*')))==100

WDIR = Path('baselines/Retinexformer/pretrained_weights')
cands = list(WDIR.glob('**/LOL_v2_synthetic.pth')) if WDIR.exists() else []
if not cands:
    subprocess.run(['gdown','--folder','1ynK5hfQachzc8y96ZumhkPPDXzHJwaQV','-O','/content/rf_weights'],check=True)
    cands = list(Path('/content/rf_weights').glob('**/LOL_v2_synthetic.pth'))
assert cands, 'LOL_v2_synthetic.pth not found'
W = str(cands[0]); print('weights:', W)


## 3 screening runs: control vs learned noise at two scales

Same seed, data, crops, LR and 20k iterations. Control = pretrained model with no new modules.
Learned variants use `--learned-noise --noise-scale 0.5 / 1.0`. Selection = best validation PSNR.


In [ ]:
RUNS = Path('runs/learned_screen')
common = ['--loss','l1','--iters','20000','--lr','2e-5','--batch','8','--crop','128','--seed','42',
          '--pretrained',W,'--opt',OPT,'--repo','baselines/Retinexformer',
          '--data',str(DATA/'Train'),'--device','cuda',
          '--val-fraction','0.1','--val-seed','2026','--val-every','2000',
          '--checkpoint-every','5000','--log-every','500']

tasks = [('control', None), ('learned_s0.5', 0.5), ('learned_s1.0', 1.0)]
for name, scale in tasks:
    out = RUNS/name; out.mkdir(parents=True,exist_ok=True)
    if (out/'metrics.json').exists():
        print('already done:', out); continue
    cmd = ['timeout','9000',sys.executable,'scripts/finetune.py']+common+['--out',str(out)]
    if scale is not None:
        cmd += ['--learned-noise','--noise-scale',str(scale)]
    if (out/'latest_state.pth').exists(): cmd += ['--resume']
    with (out/'run.log').open('a') as log:
        rc = subprocess.run(cmd,stdout=log,stderr=subprocess.STDOUT).returncode
    print(name,'exit',rc)
    if rc not in (0,124): raise RuntimeError((out/'run.log').read_text()[-3000:])

# zip partial state BEFORE reading metrics (same safety fix as the main notebook)
with zipfile.ZipFile('/content/learned_screen_state.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for p in RUNS.rglob('*'):
        if p.is_file(): zf.write(p,p.as_posix())
print('saved /content/learned_screen_state.zip')

scores = {}
for name,_ in tasks:
    m = RUNS/name/'metrics.json'
    if m.exists(): scores[name]=json.loads(m.read_text())['best_val_psnr']
print('validation PSNR:',scores)
BEST = max(scores,key=scores.get) if scores else None
print('SELECTED ON VALIDATION:', BEST)
Path('/content/selection.json').write_text(json.dumps({'scores':scores,'best':BEST},indent=2))


## Done

Download `learned_screen_state.zip` and `selection.json`. The selected variant moves to the Kaggle
150k full run; the held-out test set is evaluated only there, once, against the matched control.
